[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/calculus_optimization/01_derivatives_and_gradients_for_ml/first_principles.ipynb)

# Topic 01: Derivatives and Gradients for Machine Learning

## 1. First-Principles Intuition & Motivation

Machine learning is, at its computational core, the repeated answering of one question: *if I nudge this parameter a tiny bit, does the loss go up or down, and by how much?*

For a single parameter, the answer is the classical derivative — the limit of the ratio of output change to input change. For a model with $d$ parameters, we need $d$ such answers at once, plus a way to combine them into a single most useful direction. The gradient is exactly that package: a vector whose components are the per-parameter sensitivities and whose *direction* turns out to be geometrically special.

### Why linearization is the right abstraction

Nonlinear functions are hard; linear functions are easy. The entire strategy of differential calculus is to replace a nonlinear function, *near one point*, by the linear function that hugs it most tightly:

$$
f(x+h) = f(x) + \nabla f(x)^\top h + o(\lVert h \rVert)
$$

Everything a first-order training algorithm does — gradient descent steps, learning-rate reasoning, backpropagation — is a manipulation of this local linear model. The error term $o(\lVert h \rVert)$ is the fine print: the model is trustworthy only for small steps, which is precisely why learning rates exist (Topic 03) and why curvature corrections help (Topic 02).

### The cast of derivative objects in ML

| Object | Symbol | Shape | Role in ML |
|---|---|---|---|
| Derivative | $f'(x)$ | scalar | Sensitivity of a scalar output to a scalar input |
| Gradient | $\nabla f(x)$ | $d \times 1$ | Steepest-ascent direction of the loss |
| Jacobian | $J_g(x)$ | $m \times d$ | Linear map of a layer $g: \mathbb{R}^d \to \mathbb{R}^m$ |
| Hessian | $\nabla^2 f(x)$ | $d \times d$ | Curvature; conditioning of the landscape |
| Subgradient | $s \in \partial f(x)$ | $d \times 1$ | Descent signal at kinks (ReLU, $L^1$) |

Each row is a different answer to "how does the output respond to the input", tailored to the shapes of input and output.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 2.1 (Derivative).** For $f: \mathbb{R} \to \mathbb{R}$, the derivative at $x$ is

$$
f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h},
$$

provided the limit exists. Equivalently, $f$ is differentiable at $x$ if there exists a number $c$ with $f(x+h) = f(x) + ch + o(h)$ as $h \to 0$; then $c = f'(x)$.

**Definition 2.2 (Partial derivative).** For $f: \mathbb{R}^d \to \mathbb{R}$ and standard basis vector $e_i$,

$$
\frac{\partial f}{\partial x_i}(x) = \lim_{h \to 0} \frac{f(x + h e_i) - f(x)}{h}.
$$

**Definition 2.3 (Total differentiability and gradient).** $f: \mathbb{R}^d \to \mathbb{R}$ is (totally, or Fréchet) differentiable at $x$ if there exists a vector $g \in \mathbb{R}^d$ such that

$$
\lim_{h \to 0} \frac{f(x+h) - f(x) - g^\top h}{\lVert h \rVert} = 0.
$$

The vector $g$ is unique and is called the gradient, written $\nabla f(x)$. When $f$ is differentiable, $\nabla f(x) = \left( \frac{\partial f}{\partial x_1}, \dots, \frac{\partial f}{\partial x_d} \right)^\top$.

**Definition 2.4 (Directional derivative).** For a unit vector $v \in \mathbb{R}^d$,

$$
D_v f(x) = \lim_{t \to 0} \frac{f(x + t v) - f(x)}{t}.
$$

If $f$ is differentiable at $x$, then $D_v f(x) = \nabla f(x)^\top v$.

**Definition 2.5 (Jacobian).** For $g: \mathbb{R}^d \to \mathbb{R}^m$ with components $g_1, \dots, g_m$, the Jacobian at $x$ is the $m \times d$ matrix

$$
J_g(x) = \begin{bmatrix} \frac{\partial g_1}{\partial x_1} & \cdots & \frac{\partial g_1}{\partial x_d} \\ \vdots & \ddots & \vdots \\ \frac{\partial g_m}{\partial x_1} & \cdots & \frac{\partial g_m}{\partial x_d} \end{bmatrix}.
$$

**Definition 2.6 (Hessian).** For twice-differentiable $f: \mathbb{R}^d \to \mathbb{R}$, the Hessian is the $d \times d$ matrix $\nabla^2 f(x)$ with entries $[\nabla^2 f(x)]_{ij} = \frac{\partial^2 f}{\partial x_i \partial x_j}(x)$. By Clairaut's theorem (continuous second partials), the Hessian is symmetric.

**Definition 2.7 (Subgradient and subdifferential).** For convex $f: \mathbb{R}^d \to \mathbb{R}$, a vector $s$ is a subgradient at $x$ if $f(z) \ge f(x) + s^\top (z - x)$ for all $z \in \mathbb{R}^d$. The set of all subgradients at $x$ is the subdifferential $\partial f(x)$.

**Theorem 2.8 (Steepest ascent).** If $f$ is differentiable at $x$ and $\nabla f(x) \neq 0$, then over all unit vectors $v$, the directional derivative $D_v f(x)$ is maximized by $v^\star = \nabla f(x)/\lVert \nabla f(x) \rVert_2$, with maximum value $\lVert \nabla f(x) \rVert_2$, and minimized by $-v^\star$.

**Theorem 2.9 (Chain rule).** If $g: \mathbb{R}^d \to \mathbb{R}^m$ is differentiable at $x$ and $f: \mathbb{R}^m \to \mathbb{R}^p$ is differentiable at $g(x)$, then $f \circ g$ is differentiable at $x$ and

$$
J_{f \circ g}(x) = J_f(g(x))\, J_g(x).
$$

For scalar-valued $f$ ($p = 1$), transposing gives the backpropagation form $\nabla_x (f \circ g)(x) = J_g(x)^\top \nabla f(g(x))$.

**Theorem 2.10 (Gradient–level-set orthogonality).** If $f$ is continuously differentiable and $r: (-\epsilon, \epsilon) \to \mathbb{R}^d$ is a differentiable curve lying in the level set $\{x : f(x) = c\}$, then $\nabla f(r(t)) \perp r'(t)$ for every $t$.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 3.1: The gradient is the steepest-ascent direction

*Claim.* For unit vectors $v$, $D_v f(x) = \nabla f(x)^\top v$ is maximized at $v^\star = \nabla f(x)/\lVert \nabla f(x) \rVert_2$.

*Proof.* Write $g = \nabla f(x) \neq 0$. By the Cauchy–Schwarz inequality, for any $v$ with $\lVert v \rVert_2 = 1$,

$$
\lvert g^\top v \rvert \le \lVert g \rVert_2 \lVert v \rVert_2 = \lVert g \rVert_2,
$$

so $-\lVert g \rVert_2 \le D_v f(x) \le \lVert g \rVert_2$. Equality in Cauchy–Schwarz holds if and only if $v$ is a scalar multiple of $g$. The two unit multiples are $\pm g/\lVert g \rVert_2$. Substituting $v^\star = g/\lVert g \rVert_2$ gives

$$
D_{v^\star} f(x) = \frac{g^\top g}{\lVert g \rVert_2} = \lVert g \rVert_2,
$$

attaining the upper bound; likewise $-v^\star$ attains the lower bound $-\lVert g \rVert_2$. $\blacksquare$

*ML consequence.* Among all directions of fixed (Euclidean) length, stepping along $-\nabla f$ decreases $f$ fastest to first order — the defining move of gradient descent.

### Proof 3.2: The vector chain rule (backpropagation identity)

*Claim.* $J_{f \circ g}(x) = J_f(g(x)) J_g(x)$ for differentiable $g: \mathbb{R}^d \to \mathbb{R}^m$, $f: \mathbb{R}^m \to \mathbb{R}^p$.

*Proof.* Let $A = J_g(x)$ and $B = J_f(y)$ with $y = g(x)$. Differentiability gives the two expansions

$$
g(x+h) = g(x) + A h + r_1(h), \qquad f(y+k) = f(y) + B k + r_2(k),
$$

where $\lVert r_1(h) \rVert = o(\lVert h \rVert)$ and $\lVert r_2(k) \rVert = o(\lVert k \rVert)$. Substituting $k = Ah + r_1(h)$,

$$
f(g(x+h)) = f(y) + B\big(A h + r_1(h)\big) + r_2\big(A h + r_1(h)\big) = f(g(x)) + BA\,h + \underbrace{B\,r_1(h) + r_2(Ah + r_1(h))}_{\rho(h)}.
$$

It remains to show $\lVert \rho(h) \rVert = o(\lVert h \rVert)$. First, $\lVert B r_1(h) \rVert \le \lVert B \rVert_{\mathrm{op}} \lVert r_1(h) \rVert = o(\lVert h \rVert)$. Second, $\lVert Ah + r_1(h) \rVert \le (\lVert A \rVert_{\mathrm{op}} + 1)\lVert h \rVert$ for small $h$, so $r_2(Ah + r_1(h)) = o(\lVert h \rVert)$ as well. Hence $f \circ g$ is differentiable at $x$ with Jacobian $BA$. $\blacksquare$

*Backprop form.* For scalar loss $L = f(g(x))$, transpose: $\nabla_x L = A^\top B^\top = J_g(x)^\top \nabla f(g(x))$. Chaining this through $L = f_k \circ f_{k-1} \circ \cdots \circ f_1$ and multiplying vector-by-matrix from the loss backwards is exactly reverse-mode automatic differentiation.

### Proof 3.3: Gradients are orthogonal to level sets

*Claim.* If $r(t)$ is a differentiable curve with $f(r(t)) = c$ for all $t$, then $\nabla f(r(t))^\top r'(t) = 0$.

*Proof.* Define $\phi(t) = f(r(t))$. Since $\phi$ is constant ($\phi \equiv c$), $\phi'(t) = 0$. By the chain rule (Proof 3.2 with $m = d$, $p = 1$ and inner map $t \mapsto r(t)$),

$$
0 = \phi'(t) = \nabla f(r(t))^\top r'(t).
$$

Since $r'(t)$ is an arbitrary tangent vector to the level set at $r(t)$, the gradient is orthogonal to every such tangent, i.e. normal to the level set. $\blacksquare$

*ML consequence.* On a loss contour plot, $-\nabla L$ crosses contours at right angles. Elongated (ill-conditioned) contours therefore force the orthogonal descent direction to zigzag — the geometric seed of Topic 03's condition-number analysis.

### Proof 3.4: The least-squares gradient and Hessian

*Claim.* For $L(w) = \frac{1}{n}\lVert Xw - y \rVert_2^2$ with $X \in \mathbb{R}^{n \times d}$,

$$
\nabla_w L(w) = \frac{2}{n} X^\top (Xw - y), \qquad \nabla_w^2 L(w) = \frac{2}{n} X^\top X.
$$

*Derivation.* Expand the squared norm using $\lVert u \rVert_2^2 = u^\top u$:

$$
L(w) = \frac{1}{n}\left( w^\top X^\top X w - 2 y^\top X w + y^\top y \right).
$$

Apply the two matrix-calculus identities $\nabla_w (w^\top A w) = (A + A^\top) w$ and $\nabla_w (b^\top w) = b$. Here $A = X^\top X$ is symmetric, so $\nabla_w (w^\top X^\top X w) = 2 X^\top X w$, and $\nabla_w (y^\top X w) = X^\top y$. Therefore

$$
\nabla_w L(w) = \frac{1}{n}\left( 2 X^\top X w - 2 X^\top y \right) = \frac{2}{n} X^\top (Xw - y).
$$

Differentiating once more, the gradient is affine in $w$ with coefficient matrix $\frac{2}{n} X^\top X$, which is the Hessian. For any $v$, $v^\top X^\top X v = \lVert X v \rVert_2^2 \ge 0$, so the Hessian is positive semidefinite — and positive definite exactly when $X$ has full column rank ($Xv = 0$ only for $v = 0$). $\blacksquare$

*ML consequence.* Linear regression is a convex problem (Topic 04); its curvature is the data Gram matrix, whose eigenvalue spread sets the gradient-descent learning-rate window (Topic 03).

### Proof 3.5: The subgradient inequality certifies global information

*Claim.* If $f$ is convex and $0 \in \partial f(x^\star)$, then $x^\star$ is a global minimizer. Moreover, for differentiable convex $f$, $\partial f(x) = \{\nabla f(x)\}$.

*Proof.* If $0 \in \partial f(x^\star)$, the subgradient inequality with $s = 0$ reads

$$
f(z) \ge f(x^\star) + 0^\top (z - x^\star) = f(x^\star) \quad \text{for all } z,
$$

which is precisely global minimality. For the second statement: if $f$ is differentiable and convex, the first-order characterization of convexity (proved in Topic 04) gives $f(z) \ge f(x) + \nabla f(x)^\top (z-x)$ for all $z$, so $\nabla f(x) \in \partial f(x)$. Conversely, if $s \in \partial f(x)$, then for any direction $u$ and $t \gt 0$, $f(x + tu) - f(x) \ge t\, s^\top u$; dividing by $t$ and letting $t \to 0^+$ yields $\nabla f(x)^\top u \ge s^\top u$ for all $u$, and applying this to $u$ and $-u$ forces $s = \nabla f(x)$. $\blacksquare$

*Worked example.* $f(x) = \lvert x \rvert$ has $\partial f(0) = [-1, 1]$: for any $s \in [-1,1]$, $\lvert z \rvert \ge s z$ holds for all $z$. This is why training with ReLU or $L^1$ penalties is principled despite kinks.

### Proof 3.6: Central differences have second-order accuracy

*Claim.* For $f \in C^3$, the central-difference estimate satisfies

$$
\frac{f(x+\epsilon) - f(x-\epsilon)}{2\epsilon} = f'(x) + \frac{f'''(\xi)}{6}\epsilon^2 \quad \text{for some } \xi \in (x-\epsilon,\, x+\epsilon).
$$

*Proof.* Taylor's theorem with Lagrange remainder (proved in Topic 02) gives

$$
f(x+\epsilon) = f(x) + f'(x)\epsilon + \frac{f''(x)}{2}\epsilon^2 + \frac{f'''(\xi_1)}{6}\epsilon^3, \qquad f(x-\epsilon) = f(x) - f'(x)\epsilon + \frac{f''(x)}{2}\epsilon^2 - \frac{f'''(\xi_2)}{6}\epsilon^3.
$$

Subtracting, the even-order terms cancel:

$$
f(x+\epsilon) - f(x-\epsilon) = 2 f'(x)\epsilon + \frac{\epsilon^3}{6}\left( f'''(\xi_1) + f'''(\xi_2) \right).
$$

Divide by $2\epsilon$ and apply the intermediate value theorem to $f'''$ to replace the average $\frac{1}{2}(f'''(\xi_1) + f'''(\xi_2))$ by $f'''(\xi)$ at a single point. The error is $O(\epsilon^2)$, versus $O(\epsilon)$ for the one-sided quotient, whose $\frac{f''}{2}\epsilon$ term does not cancel. $\blacksquare$

*Practical fine print.* In floating point, subtraction of nearby values injects a round-off error of order $\epsilon_{\text{mach}} \lvert f(x) \rvert/\epsilon$. Total error is minimized near $\epsilon \sim \epsilon_{\text{mach}}^{1/3}$ for central differences — the classic U-shaped error curve.

## 4. Computational & Algorithmic Insights

### 4.1 Backpropagation is the chain rule, organized

For a deep network $L = \ell(f_k(f_{k-1}(\cdots f_1(x; w_1) \cdots; w_{k-1}); w_k))$, the chain rule gives a product of Jacobians. The *order* of multiplication is a free algorithmic choice with enormous cost consequences:

- **Forward mode** propagates $J v$ products input-to-output: one pass per *input* direction. Cost to get a full gradient of $L: \mathbb{R}^d \to \mathbb{R}$ is $O(d)$ passes — hopeless for $d \sim 10^9$.
- **Reverse mode (backprop)** propagates $J^\top u$ products output-to-input: one pass per *output*. A scalar loss needs exactly one backward pass, costing a small constant (roughly 2–3) times the forward pass, at the price of storing intermediate activations.

The cheap gradient principle (Baydin et al., 2018; Griewank & Walther, 2008) — a full gradient for a constant-factor overhead — is arguably the single result that makes deep learning computationally feasible.

### 4.2 Forward mode via dual numbers

Forward-mode autodiff can be implemented by extending arithmetic to dual numbers $a + b\epsilon$ with $\epsilon^2 = 0$. Every elementary operation propagates the pair (value, derivative):

$$
(a + b\epsilon)(c + d\epsilon) = ac + (ad + bc)\epsilon, \qquad f(a + b\epsilon) = f(a) + f'(a) b \epsilon.
$$

Running $f$ on $x + 1 \cdot \epsilon$ returns $f(x) + f'(x)\epsilon$: the exact derivative rides along with the value. This is the method of choice for Jacobian-vector products, e.g. in Hessian-vector computations $\nabla^2 f(x) v$ obtained by forward-over-reverse composition.

### 4.3 Gradient checking protocol

When implementing gradients by hand, verify against central differences componentwise, using the relative criterion

$$
\frac{\lVert g - \hat{g} \rVert_2}{\lVert g \rVert_2 + \lVert \hat{g} \rVert_2} \lt 10^{-5},
$$

with $\epsilon \sim 10^{-5}$ to $10^{-7}$ in double precision. Checks should be run at *random* points (not symmetric special points where errors cancel) and before any performance optimization. Common failures: forgetting the transpose in $J^\top$, dropping a $\frac{2}{n}$ factor, and testing at $x = 0$ where kinks hide.

### 4.4 What each object costs

| Quantity | Exact cost (relative to 1 forward pass) | Typical ML use |
|---|---|---|
| Loss value $L(w)$ | 1 | Monitoring |
| Full gradient $\nabla L$ | 2–3 (reverse mode) | Every SGD/Adam step |
| Jacobian-vector product $Jv$ | 2 (forward mode) | Sensitivity, tangent propagation |
| Vector-Jacobian product $J^\top u$ | 2–3 (reverse mode) | Backprop building block |
| Hessian-vector product $\nabla^2 L\, v$ | 4–6 (forward-over-reverse) | Curvature probes, trust region |
| Full Hessian $\nabla^2 L$ | $O(d)$ passes | Only feasible for small $d$ |

The table explains the shape of practical ML optimization: first-order methods with occasional Hessian-vector probes, never explicit Hessians at scale.

## 5. Real-World Physics & AI/ML Applications

### 5.1 Physics: potentials, forces, and gradient fields

In classical mechanics, a conservative force is the negative gradient of a potential energy: $F = -\nabla U$. A ball rolling in a bowl follows the steepest-descent direction of $U$ — gradient descent on the loss surface is the exact computational analog, with the loss playing the role of potential energy. Level-set orthogonality (Proof 3.3) is the statement that forces act perpendicular to equipotential surfaces. In thermodynamics and electrostatics the same structure recurs: heat flows along $-\nabla T$, the electric field is $E = -\nabla V$.

### 5.2 ML: training loops as gradient consumers

- **Linear/logistic regression**: closed-form gradients (Proof 3.4 and its logistic analog $\nabla L = \frac{1}{n} X^\top(\sigma(Xw) - y)$) feed directly into gradient descent.
- **Deep networks**: reverse-mode autodiff computes $\nabla_w L$ layer by layer; frameworks like PyTorch and JAX build the computation graph and apply Theorem 2.9 mechanically.
- **Non-smooth training**: ReLU networks and $L^1$-regularized models (Lasso) rely on subgradients; proximal methods refine this by treating the non-smooth part exactly.
- **Gradient checking**: still the standard unit test when writing custom layers, losses, or CUDA kernels.
- **Adversarial examples and saliency**: the *input* gradient $\nabla_x L$ (same machinery, different variable) identifies the input directions the model is most sensitive to.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source | Where |
|---|---|---|
| Gradients, Jacobians, Hessians for ML | Goodfellow, Bengio & Courville, *Deep Learning* | Ch. 4.2–4.3 |
| Backpropagation as chain rule | Goodfellow et al., *Deep Learning* | Ch. 6.5 |
| Forward vs reverse autodiff, cheap gradient principle | Baydin et al., *Automatic Differentiation in ML: a Survey* (JMLR 2018) | §2–3 |
| Rigorous total derivative and chain rule | Spivak, *Calculus on Manifolds* | Ch. 2 |
| Matrix calculus identities | Petersen & Pedersen, *The Matrix Cookbook* | §2 |
| Subgradients and subdifferentials | Rockafellar, *Convex Analysis*; Boyd & Vandenberghe, *Convex Optimization* | Part V; Ch. 3 |
| Finite differences and derivative computation | Nocedal & Wright, *Numerical Optimization* | Ch. 8 |
| Algorithmic differentiation in depth | Griewank & Walther, *Evaluating Derivatives* | Ch. 3–4 |

**Forward pointers within this module**: Topic 02 turns the gradient into quantitative local models with error bounds; Topic 03 analyzes the algorithm that consumes gradients; Topic 04 classifies the landscapes gradients navigate. The sibling [`optimization/`](../../optimization/) module develops general optimization theory beyond this calculus bridge.